# HELM Benchmark-wise Calibration

This notebook fits the model separately for each HELM benchmark with early stopping and logs metrics.

# Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pickle
import pandas as pd
import warnings
import json
import os
from collections import defaultdict
from sklearn.metrics import roc_auc_score, accuracy_score
from hypothesaes.quickstart import train_sae
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Fixed seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Reproducibility seed: {SEED}")

Using device: cuda


# Load and Align Data

In [10]:
print("Loading data...")
y_df = pd.read_pickle('../data-reeval-multi/resmat.pkl')
emb_df = pd.read_pickle('../data/embed_meta-llama_Llama-3.1-8B-Instruct.pkl')

# Filter Rows/Cols in y_df
y_df = y_df[y_df.notna().any(axis=1)]
valid_cols_list = []
for c in y_df.columns:
    valid_cols_list.append(y_df[c].notna().any() and (y_df[c].dropna() != 0).any())
y_df = y_df.iloc[:, valid_cols_list]
print(f"Target Matrix Shape: {y_df.shape}")

# Alignment Logic
print("\nAligning Embeddings to Question Text...")
if 'question' not in emb_df.columns:
    text_col = [c for c in emb_df.columns if 'text' in str(c) or 'question' in str(c)][0]
    emb_df = emb_df.rename(columns={text_col: 'question'})

# Create embedding lookup
emb_map = {}
for _, row in emb_df.iterrows():
    q_text = row['question']
    emb = row['embedding']
    if isinstance(emb, str):
        import ast
        emb = ast.literal_eval(emb)
    emb_map[q_text] = emb

# Extract questions from y_df columns
if isinstance(y_df.columns, pd.MultiIndex):
    if 'input.text' in y_df.columns.names:
        questions = y_df.columns.get_level_values('input.text').tolist()
    else:
        questions = y_df.columns.get_level_values(-1).tolist()
else:
    questions = y_df.columns.tolist()

# Build aligned lists
aligned_raw_embs = []
valid_indices_mask = []
for q in questions:
    if q in emb_map:
        aligned_raw_embs.append(emb_map[q])
        valid_indices_mask.append(True)
    else:
        valid_indices_mask.append(False)

valid_indices_mask = np.array(valid_indices_mask)
print(f"Found embeddings for {valid_indices_mask.sum()} / {len(questions)} items")

# Filter y_df based on alignment
y_df = y_df.iloc[:, valid_indices_mask]
x_j_dense = torch.tensor(np.array(aligned_raw_embs), dtype=torch.float32)
x_j_dense = F.normalize(x_j_dense, p=2, dim=1).to(device)

print(f"\nFinal aligned data shape: {y_df.shape}")
print(f"Embeddings shape: {x_j_dense.shape}")

Loading data...


Target Matrix Shape: (183, 78712)

Aligning Embeddings to Question Text...
Found embeddings for 78688 / 78712 items

Final aligned data shape: (183, 78688)
Embeddings shape: torch.Size([78688, 4096])


# Extract Scenarios and Prepare Global SAE

In [11]:
# Get unique scenarios
if isinstance(y_df.columns, pd.MultiIndex) and 'scenario' in y_df.columns.names:
    scenarios = y_df.columns.get_level_values('scenario').unique().tolist()
else:
    # Fallback: treat as single scenario
    scenarios = ['all_data']

print(f"Found {len(scenarios)} scenarios/benchmarks:")
for scenario in scenarios:
    if isinstance(y_df.columns, pd.MultiIndex) and 'scenario' in y_df.columns.names:
        mask = y_df.columns.get_level_values('scenario') == scenario
        n_items = mask.sum()
        n_models = y_df.loc[:, mask].notna().any(axis=1).sum()
    else:
        n_items = y_df.shape[1]
        n_models = y_df.notna().any(axis=1).sum()
    print(f"  {scenario}: {n_models} models, {n_items} items")

# Train SAE on all embeddings (global feature extractor)
print("\nTraining SAE on all embeddings...")
embeddings_np = x_j_dense.cpu().numpy()

sae = train_sae(
    embeddings=embeddings_np,
    M=1024,
    K=32,
    batch_size=512,
    n_epochs=50,
    learning_rate=5e-4,
    checkpoint_dir='checkpoints/helm_sae'
)

# Transform to sparse activations
print("Transforming embeddings to SAE activations...")
sae_activations_np = sae.get_activations(embeddings_np)
x_j_global = torch.tensor(sae_activations_np, dtype=torch.float32).to(device)
d_features = sae.m_total_neurons
print(f"SAE Feature Dimension: {d_features}")

Found 22 scenarios/benchmarks:
  lsat_qa: 69 models, 454 items
  truthful_qa: 67 models, 1888 items
  synthetic_reasoning: 69 models, 2234 items
  babi_qa: 70 models, 3461 items
  wikifact: 67 models, 5511 items
  bbq: 42 models, 999 items
  thai_exam: 40 models, 557 items
  dyck_language_np=3: 69 models, 500 items
  legal_support: 69 models, 594 items
  civil_comments: 67 models, 29407 items
  legalbench: 91 models, 1990 items
  raft: 67 models, 1330 items
  air_bench_2024: 41 models, 4985 items
  math: 91 models, 436 items
  med_qa: 91 models, 998 items
  gsm: 90 models, 997 items
  boolq: 67 models, 3316 items
  mmlu: 79 models, 13223 items
  entity_matching: 67 models, 1396 items
  entity_data_imputation: 67 models, 395 items
  commonsense: 91 models, 498 items
  imdb: 67 models, 3519 items

Training SAE on all embeddings...
Loaded model from checkpoints/helm_sae/SAE_M=1024_K=32.pt onto device cuda
Transforming embeddings to SAE activations...


Computing activations (batchsize=16384):   0%|          | 0/5 [00:00<?, ?it/s]

SAE Feature Dimension: 1024


# Define Model Architecture

In [12]:
class LinearRobustARD(nn.Module):
    def __init__(self, N, J, K_model, d_features, x_j_input):
        super().__init__()
        self.N, self.J, self.K = N, J, K_model
        self.register_buffer('x_j', x_j_input)

        # Latent Factors
        self.theta = nn.Parameter(torch.randn(N, K_model) * 0.1)
        self.W = nn.Parameter(torch.randn(K_model, d_features) * 0.01)
        self.tau_raw = nn.Parameter(torch.ones(K_model) * 0.5)
        
        # Linear Amortized Difficulty
        self.difficulty_proj = nn.Linear(d_features, 1)

    @property
    def tau(self):
        return F.relu(self.tau_raw)

    def forward(self):
        x_j = self.x_j
        
        # Linear Difficulty Projection
        pred_delta = self.difficulty_proj(x_j).squeeze().unsqueeze(0)
        
        # Linear Loading Projection
        W_norm = F.normalize(self.W, dim=1)
        a_j = (x_j @ W_norm.T) * self.tau.unsqueeze(0)

        # Overall Prediction
        logits_y = self.theta @ a_j.T + pred_delta
        return logits_y

# Benchmark-wise Training with Early Stopping

In [ ]:
# Benchmark-specific hyperparameters
# Add benchmarks here to calibrate them with specific K_MODEL and lambda_tau
benchmark_params = {
    # 'lsat_qa': {'K_MODEL': 50, 'lambda_tau': 30.0},
    # 'truthful_qa': {'K_MODEL': 100, 'lambda_tau': 30.0},
    # 'synthetic_reasoning': {'K_MODEL': 50, 'lambda_tau': 25.0},
    'babi_qa': {'K_MODEL': 50, 'lambda_tau': 50.0},
    # 'wikifact': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'bbq': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'thai_exam': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'dyck_language_np=3': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'legal_support': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'civil_comments': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'legalbench': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'raft': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'air_bench_2024': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'math': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'med_qa': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'gsm': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'boolq': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'mmlu': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'entity_matching': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'entity_data_imputation': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'commonsense': {'K_MODEL': 100, 'lambda_tau': 1500.0},
    # 'imdb': {'K_MODEL': 100, 'lambda_tau': 1500.0},
}

# Shared hyperparameters
reg_theta = 0.5
lr_tau = 0.01
lr_proj = 0.005
lr_latent = 0.01
wd_proj = 1e-2
wd_latent = 1e-4
max_epochs = 2000
patience = 50
min_delta = 1e-5

# Results storage
metric_results = defaultdict(dict)
os.makedirs('../result', exist_ok=True)

# Filter scenarios to only those in benchmark_params
scenarios_to_calibrate = [s for s in scenarios if s in benchmark_params]
print(f"\nCalibrating {len(scenarios_to_calibrate)} benchmarks out of {len(scenarios)} total")
print(f"Benchmarks to calibrate: {scenarios_to_calibrate}")

# Train each scenario separately
for scenario in tqdm(scenarios_to_calibrate, desc="Processing scenarios"):
    # Get benchmark-specific parameters
    K_MODEL = benchmark_params[scenario]['K_MODEL']
    lambda_tau = benchmark_params[scenario]['lambda_tau']
    
    print(f"\n{'='*60}")
    print(f"Training on scenario: {scenario}")
    print(f"K_MODEL: {K_MODEL}, lambda_tau: {lambda_tau}")
    print(f"{'='*60}")
    
    # Extract scenario-specific data
    if isinstance(y_df.columns, pd.MultiIndex) and 'scenario' in y_df.columns.names:
        mask = y_df.columns.get_level_values('scenario') == scenario
        y_scenario = y_df.loc[:, mask]
        x_j_scenario = x_j_global[mask]
    else:
        y_scenario = y_df
        x_j_scenario = x_j_global
    
    # Convert to arrays
    y_vals = y_scenario.values.astype(np.float32)
    N, J = y_vals.shape
    
    print(f"Scenario shape: {N} models x {J} items")
    
    # Create item-wise train/test split (cold start) with fixed seed for reproducibility
    np.random.seed(SEED)
    J_indices = np.arange(J)
    np.random.shuffle(J_indices)
    n_test = int(0.1 * J)
    test_idx = J_indices[:n_test]
    train_idx = J_indices[n_test:]
    
    train_mask = np.zeros_like(y_vals, dtype=bool)
    train_mask[:, train_idx] = ~np.isnan(y_vals)[:, train_idx]
    
    test_mask = np.zeros_like(y_vals, dtype=bool)
    test_mask[:, test_idx] = ~np.isnan(y_vals)[:, test_idx]
    
    y_data = torch.from_numpy(np.nan_to_num(y_vals, nan=0.0)).to(device)
    train_mask = torch.from_numpy(train_mask).to(device)
    test_mask = torch.from_numpy(test_mask).to(device)
    
    # Initialize model with fixed seed for reproducibility
    torch.manual_seed(SEED)
    model = LinearRobustARD(N, J, K_MODEL, d_features, x_j_scenario).to(device)
    
    # Optimizers
    opt_local = optim.Adam(
        [{'params': [model.theta], 'lr': lr_latent, 'weight_decay': wd_latent}],
        lr=lr_latent
    )
    
    opt_global = optim.Adam([
        {'params': model.tau_raw, 'lr': lr_tau, 'weight_decay': 0.0},
        {'params': [model.W], 'lr': lr_proj, 'weight_decay': wd_proj},
        {'params': list(model.difficulty_proj.parameters()), 'lr': lr_proj, 'weight_decay': wd_proj}
    ], lr=lr_proj)
    
    # Training with early stopping
    best_test_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(max_epochs):
        model.train()
        
        # Local update (theta)
        opt_local.zero_grad()
        logits_y = model()
        lik_y = (F.binary_cross_entropy_with_logits(logits_y, y_data, reduction='none') * train_mask).sum()
        reg_theta_term = reg_theta * torch.sum(model.theta**2)
        loss_local = lik_y + reg_theta_term
        loss_local.backward()
        opt_local.step()
        
        # Global update (W, difficulty_proj, tau)
        opt_global.zero_grad()
        logits_y = model()
        lik_y = (F.binary_cross_entropy_with_logits(logits_y, y_data, reduction='none') * train_mask).sum()
        reg_tau = lambda_tau * torch.norm(model.tau, 1)
        loss_global = lik_y + reg_tau
        
        if torch.isnan(loss_global):
            print("WARNING: Loss is NaN! Stopping this scenario.")
            break
        
        loss_global.backward()
        opt_global.step()
        
        with torch.no_grad():
            model.tau_raw[model.tau < 0.01] = -0.1
        
        # Early stopping check
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                logits_y = model()
                test_loss = (F.binary_cross_entropy_with_logits(logits_y, y_data, reduction='none') * test_mask).sum()
                
                if test_loss < best_test_loss - min_delta:
                    best_test_loss = test_loss
                    patience_counter = 0
                else:
                    patience_counter += 1
                
                if epoch % 100 == 0:
                    active_dims = (model.tau > 0.01).sum().item()
                    print(f"Epoch {epoch} | Train Loss: {loss_global.item():.2e} | Test Loss: {test_loss.item():.2e} | Active Dims: {active_dims} | Patience: {patience_counter}")
                
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break
    
    # Evaluation
    print(f"\nEvaluating {scenario}...")
    model.eval()
    with torch.no_grad():
        logits_y = model()
        probs = torch.sigmoid(logits_y)
        
        # Test metrics
        y_test = torch.masked_select(y_data, test_mask).cpu().numpy()
        p_test = torch.masked_select(probs, test_mask).cpu().numpy()
        
        # Train metrics
        y_train = torch.masked_select(y_data, train_mask).cpu().numpy()
        p_train = torch.masked_select(probs, train_mask).cpu().numpy()
        
        # Calculate metrics
        test_auc = roc_auc_score(y_test, p_test) if len(np.unique(y_test)) > 1 else 0.0
        test_acc = accuracy_score(y_test, (p_test > 0.5).astype(int))
        train_auc = roc_auc_score(y_train, p_train) if len(np.unique(y_train)) > 1 else 0.0
        train_acc = accuracy_score(y_train, (p_train > 0.5).astype(int))
        
        # Store results
        metric_results[scenario]['train_auc'] = float(train_auc)
        metric_results[scenario]['train_acc'] = float(train_acc)
        metric_results[scenario]['test_auc'] = float(test_auc)
        metric_results[scenario]['test_acc'] = float(test_acc)
        metric_results[scenario]['n_models'] = int(N)
        metric_results[scenario]['n_items'] = int(J)
        metric_results[scenario]['n_test_items'] = int(n_test)
        metric_results[scenario]['final_epoch'] = epoch
        
        print(f"Results for {scenario}:")
        print(f"  Train AUC: {train_auc:.4f} | Train Acc: {train_acc:.4f}")
        print(f"  Test AUC:  {test_auc:.4f} | Test Acc:  {test_acc:.4f}")
        print(f"  Test set size: {len(y_test)} responses")
    
    # Cleanup
    del model, opt_local, opt_global
    torch.cuda.empty_cache()

# Save results
with open('../result/helm_calibration_results.json', 'w') as f:
    json.dump(metric_results, f, indent=4)

print("\n" + "="*60)
print("All scenarios completed! Results saved.")
print("="*60)


Calibrating 1 benchmarks out of 22 total
Benchmarks to calibrate: ['babi_qa']


Processing scenarios:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      | 0/1 [00:00<?, ?it/s]


Training on scenario: babi_qa
K_MODEL: 50, lambda_tau: 50.0
Scenario shape: 183 models x 3461 items
Epoch 0 | Train Loss: 1.54e+05 | Test Loss: 1.68e+04 | Active Dims: 50 | Patience: 0
Epoch 100 | Train Loss: 1.12e+05 | Test Loss: 1.20e+04 | Active Dims: 50 | Patience: 0
Epoch 200 | Train Loss: 1.11e+05 | Test Loss: 1.19e+04 | Active Dims: 50 | Patience: 0
Epoch 300 | Train Loss: 1.10e+05 | Test Loss: 1.18e+04 | Active Dims: 50 | Patience: 0
Epoch 400 | Train Loss: 1.09e+05 | Test Loss: 1.17e+04 | Active Dims: 50 | Patience: 0
Epoch 500 | Train Loss: 1.09e+05 | Test Loss: 1.17e+04 | Active Dims: 50 | Patience: 0
Epoch 600 | Train Loss: 1.08e+05 | Test Loss: 1.17e+04 | Active Dims: 50 | Patience: 0
Epoch 700 | Train Loss: 1.08e+05 | Test Loss: 1.17e+04 | Active Dims: 39 | Patience: 0
Epoch 800 | Train Loss: 1.08e+05 | Test Loss: 1.16e+04 | Active Dims: 28 | Patience: 0
Epoch 900 | Train Loss: 1.07e+05 | Test Loss: 1.16e+04 | Active Dims: 20 | Patience: 0
Epoch 1000 | Train Loss: 1.07e+

Processing scenarios: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:07<00:00,  7.14s/it]

Early stopping at epoch 1890

Evaluating babi_qa...
Results for babi_qa:
  Train AUC: 0.8043 | Train Acc: 0.7731
  Test AUC:  0.8122 | Test Acc:  0.7708
  Test set size: 24220 responses

All scenarios completed! Results saved.


# Calculate and Display Average Metrics

In [14]:
# # Load results if needed
# with open('../result/helm_calibration_results.json', 'r') as f:
#     metric_results = json.load(f)

# # Calculate averages
# test_aucs = [metric_results[s]['test_auc'] for s in metric_results]
# test_accs = [metric_results[s]['test_acc'] for s in metric_results]
# train_aucs = [metric_results[s]['train_auc'] for s in metric_results]
# train_accs = [metric_results[s]['train_acc'] for s in metric_results]

# avg_test_auc = np.mean(test_aucs)
# avg_test_acc = np.mean(test_accs)
# avg_train_auc = np.mean(train_aucs)
# avg_train_acc = np.mean(train_accs)

# std_test_auc = np.std(test_aucs)
# std_test_acc = np.std(test_accs)

# print("\n" + "="*60)
# print("AGGREGATE RESULTS ACROSS ALL BENCHMARKS")
# print("="*60)
# print(f"\nNumber of benchmarks: {len(metric_results)}")
# print(f"\nAverage Train AUC: {avg_train_auc:.4f}")
# print(f"Average Train Acc: {avg_train_acc:.4f}")
# print(f"\nAverage Test AUC:  {avg_test_auc:.4f} ± {std_test_auc:.4f}")
# print(f"Average Test Acc:  {avg_test_acc:.4f} ± {std_test_acc:.4f}")
# print("\n" + "="*60)

# # Create summary DataFrame
# summary_df = pd.DataFrame(metric_results).T
# summary_df = summary_df.sort_values('test_auc', ascending=False)

# print("\nPer-Benchmark Results (sorted by Test AUC):")
# print(summary_df[['test_auc', 'test_acc', 'n_models', 'n_test_items']].to_string())

# # Save summary
# summary_df.to_csv('../result/helm_calibration_summary.csv')
# print("\nSummary saved to ../result/helm_calibration_summary.csv")

# Visualize Results

In [15]:
# import matplotlib.pyplot as plt

# # Plot test AUC and ACC per benchmark
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# benchmarks = list(metric_results.keys())
# test_aucs = [metric_results[b]['test_auc'] for b in benchmarks]
# test_accs = [metric_results[b]['test_acc'] for b in benchmarks]

# # AUC plot
# ax1.barh(benchmarks, test_aucs, color='skyblue')
# ax1.axvline(avg_test_auc, color='red', linestyle='--', label=f'Average: {avg_test_auc:.3f}')
# ax1.set_xlabel('Test AUC', fontsize=12)
# ax1.set_title('Test AUC by Benchmark', fontsize=14)
# ax1.legend()
# ax1.grid(axis='x', alpha=0.3)

# # ACC plot
# ax2.barh(benchmarks, test_accs, color='lightcoral')
# ax2.axvline(avg_test_acc, color='red', linestyle='--', label=f'Average: {avg_test_acc:.3f}')
# ax2.set_xlabel('Test Accuracy', fontsize=12)
# ax2.set_title('Test Accuracy by Benchmark', fontsize=14)
# ax2.legend()
# ax2.grid(axis='x', alpha=0.3)

# plt.tight_layout()
# plt.savefig('../result/helm_calibration_benchmark_comparison.png', dpi=300, bbox_inches='tight')
# plt.show()

# print("Visualization saved to ../result/helm_calibration_benchmark_comparison.png")